# Basler Camera Capture

This notebook walks through the `physicalai.capture` API for Basler cameras:
discover devices, connect via serial number, and capture color frames.

**Prerequisites** — install the pypylon SDK bindings and matplotlib:

```bash
uv pip install physicalai[basler] matplotlib
```

## 1. Discover available devices

In [ ]:
from physicalai.capture import BaslerCamera

devices = BaslerCamera.discover()
for i, dev in enumerate(devices):
    ip_address = dev.metadata.get("address", "N/A") if dev.metadata else None
    print(f"[{i}] {dev.device_id} - {dev.name} ({dev.manufacturer} {dev.model}{'@' + ip_address if ip_address else ''})")

if not devices:
    raise RuntimeError("No Basler devices found. Check USB/GigE connection and pylon drivers.")

## 2. Connect

Pick the serial number from the list above.

In [ ]:
serial = devices[0].device_id  # change index if you have multiple cameras

cam = BaslerCamera(serial_number=serial, fps=30, width=640, height=480)
cam.connect()
print(f"Connected to {cam.device_id}")

## 3. Capture a color frame

In [ ]:
import matplotlib.pyplot as plt

frame = cam.read()
print(f"shape={frame.data.shape}  dtype={frame.data.dtype}  seq={frame.sequence}")

plt.figure(figsize=(8, 6))
plt.imshow(frame.data)
plt.title("Color")
plt.axis("off")
plt.show()

## 4. Grab the latest frame (non-blocking)

`read_latest()` returns the most recent frame without waiting for a new one — useful
for control loops that must not block on the camera.

In [ ]:
latest = cam.read_latest()
print(f"shape={latest.data.shape}  dtype={latest.data.dtype}  seq={latest.sequence}")

plt.figure(figsize=(8, 6))
plt.imshow(latest.data)
plt.title(f"Latest  seq={latest.sequence}")
plt.axis("off")
plt.show()

## 5. Stream a short burst

Read a handful of frames back-to-back and display them as a grid.

In [ ]:
n = 6
frames = [cam.read() for _ in range(n)]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, f in zip(axes.flat, frames):
    ax.imshow(f.data)
    ax.set_title(f"seq={f.sequence}  t={f.timestamp:.3f}s")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. Disconnect

In [ ]:
cam.disconnect()
print(f"Connected: {cam.is_connected}")